# MACS Compressor — LSTM Byte Predictor Training

## Lane A: Text / Code Lossless Compression

This notebook trains the LSTM byte-prediction model used in `text_compressor.py`.
The trained model is saved as `backend/models/lstm_text_v1.h5`.

### Algorithm
1. A two-layer LSTM processes text bytes sequentially
2. At each position it outputs `p(next_byte | all_previous_bytes)` — a 256-dim softmax
3. These probabilities feed into the `constriction` arithmetic encoder
4. Compression length = `−Σ log₂ p(xₜ | x<ₜ)` bits (Shannon entropy lower bound)

### Model versioning
The version byte `0x01` is stored at offset 7 of every `.macs` header.
If you retrain, bump to `v2` and save as `lstm_text_v2.h5`.

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## 1. Configuration

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
SEQ_LEN      = 64      # context window: how many past bytes the LSTM sees
BATCH_SIZE   = 256
EPOCHS       = 20      # early stopping will halt before this if val_loss plateaus
LSTM_UNITS   = 256     # hidden units per LSTM layer
EMBED_DIM    = 64      # byte embedding dimension
LEARNING_RATE = 1e-3

MODEL_PATH   = '../backend/models/lstm_text_v1.h5'
MODEL_VERSION = 0x01   # must match MODEL_VERSION_V1 in text_compressor.py

os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
print(f'Model will be saved to: {os.path.abspath(MODEL_PATH)}')

## 2. Load Training Corpus

In [ ]:
import glob

# ── Collect training files ────────────────────────────────────────────────────
# Add paths to your text/code corpus below.
# For best performance: mix of English prose, Python, JavaScript, JSON, CSV.
CORPUS_GLOBS = [
    '../samples/*.txt',
    '../samples/*.py',
    '../samples/*.json',
    # Add more paths here, e.g. '/path/to/wikipedia_en/*.txt'
]

all_bytes = bytearray()
file_count = 0
for pattern in CORPUS_GLOBS:
    for filepath in glob.glob(pattern, recursive=True):
        try:
            with open(filepath, 'rb') as f:
                data = f.read()
            all_bytes.extend(data)
            file_count += 1
            print(f'  Loaded {filepath} ({len(data):,} bytes)')
        except Exception as e:
            print(f'  [warn] {filepath}: {e}')

print(f'\nTotal: {file_count} files, {len(all_bytes):,} bytes ({len(all_bytes)/1024:.1f} KB)')

if len(all_bytes) < SEQ_LEN * BATCH_SIZE:
    print('[WARNING] Corpus very small — model will overfit. Add more files for good compression.')

## 3. Prepare Dataset (Sliding Window)

In [ ]:
# Build sliding-window dataset: X[i] = bytes[i:i+SEQ_LEN], y[i] = bytes[i+SEQ_LEN]
corpus = np.frombuffer(bytes(all_bytes), dtype=np.uint8)

n_samples = len(corpus) - SEQ_LEN
print(f'Number of training samples: {n_samples:,}')

# Use tf.data for memory-efficient batching
def make_dataset(corpus_arr, seq_len, batch_size):
    dataset = tf.data.Dataset.from_tensor_slices(corpus_arr.astype(np.int32))
    # Create (context, target) windows
    windows = dataset.window(seq_len + 1, shift=1, drop_remainder=True)
    windows = windows.flat_map(lambda w: w.batch(seq_len + 1))
    
    def split_xy(window):
        return window[:-1], window[-1]
    
    return (
        windows
        .map(split_xy, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(10000)
        .batch(batch_size, drop_remainder=True)
        .prefetch(tf.data.AUTOTUNE)
    )

# 90/10 train/val split
split_idx = int(len(corpus) * 0.9)
train_corpus = corpus[:split_idx]
val_corpus   = corpus[split_idx:]

train_ds = make_dataset(train_corpus, SEQ_LEN, BATCH_SIZE)
val_ds   = make_dataset(val_corpus,   SEQ_LEN, BATCH_SIZE)

print(f'Train samples (approx): {len(train_corpus) - SEQ_LEN:,}')
print(f'Val   samples (approx): {len(val_corpus)   - SEQ_LEN:,}')

## 4. Build LSTM Model

In [ ]:
def build_lstm_model(seq_len, lstm_units, embed_dim, lr):
    model = Sequential([
        # Byte embedding: maps each byte (0–255) to a dense vector
        Embedding(input_dim=256, output_dim=embed_dim, input_length=seq_len),
        # Two LSTM layers: first returns sequences, second returns final state
        LSTM(lstm_units, return_sequences=True, dropout=0.1),
        LSTM(lstm_units, dropout=0.1),
        # Output: softmax over 256 possible next bytes
        Dense(256, activation='softmax'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',  # y is a single byte index
        metrics=['accuracy'],
    )
    return model

model = build_lstm_model(SEQ_LEN, LSTM_UNITS, EMBED_DIM, LEARNING_RATE)
model.summary()

# Theoretical compression from model perplexity:
# If val loss (cross-entropy) = H bits/byte, compression ratio ≈ 8/H
# Target: H < 3.0 bits/byte → ratio > 2.7x

## 5. Train

In [ ]:
callbacks = [
    ModelCheckpoint(
        MODEL_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 6. Evaluate & Plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Cross-Entropy Loss (bits/byte)')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['accuracy'],     label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Next-Byte Prediction Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('../notebooks/training_curves.png', dpi=150)
plt.show()

final_loss = min(history.history['val_loss'])
bits_per_byte = final_loss / 0.693  # nats → bits (ln2)
est_ratio = 8.0 / bits_per_byte
print(f'\nBest val loss:           {final_loss:.4f} nats/byte')
print(f'Estimated bits/byte:     {bits_per_byte:.2f}')
print(f'Expected compression:    {est_ratio:.1f}x')
print(f'Model saved to:          {os.path.abspath(MODEL_PATH)}')

## 7. Quick Smoke Test
Verify the saved model loads and produces valid probability distributions.

In [ ]:
from tensorflow.keras.models import load_model

loaded = load_model(MODEL_PATH, compile=False)
test_context = np.array([[ord('H'), ord('e'), ord('l'), ord('l'), ord('o')] + [0]*(SEQ_LEN-5)], dtype=np.int32)
# Reshape for model input
probs = loaded.predict(test_context, verbose=0)[0]   # shape (256,)
print(f'Output shape: {probs.shape}   (expected: (256,))')
print(f'Sum of probs: {probs.sum():.6f}   (expected: ~1.0)')
print(f'Max prob byte: {np.argmax(probs)} ({chr(np.argmax(probs))!r})')
print('\n✅ Model loads and predicts correctly — ready for text_compressor.py')

## Notes
- The model is version `0x01` — stored in `.macs` header offset 7
- If you retrain with more data, save as `lstm_text_v2.h5` and bump `MODEL_VERSION_V1` to `0x02` in `text_compressor.py`
- Without a GPU, training on the small sample corpus takes ~2 minutes; on a large corpus it may take hours
- The `zlib` fallback in `text_compressor.py` is used if this model is absent — it still achieves ~2.5x on typical text